# Linguistic marker religion — GSM8K

## SPEC
Memperbaiki notebook dan PNG berdasarkan data asli, tanpa mengubah scoring, persona, data, atau Bab VI. Seluruh data eksperimen enam model dan 17 skenario dipakai (GSM8K test penuh 1.319 soal; ETHICS 1.514 soal dari empat subset eksperimen). Seluruh respons diinventarisasi; marker hanya pada penjelasan final tersedia. Status tetap **kandidat menunggu audit manusia independen**.

## DESIGN PLAN
Pairing diverifikasi dengan isi soal kanonis dan pesan pengguna terakhir. Kamus lama dibekukan, bukan ditambah. Hasil baru ditulis ke direktori unik; gambar lama diarsipkan setelah pemeriksaan lolos. Semua implementasi pengolahan dan plotting ada dalam notebook ini; tidak ada skrip implementasi baru.

Perubahan inferensi dinyatakan eksplisit: signed-rank untuk outcome numerik, McNemar eksak untuk kehadiran, Holm seluruh outcome/kontras/scope per model. Jangan membandingkan p lama–baru tanpa memperhitungkan perubahan ini. I–E adalah perbandingan paket percakapan, bukan identitas murni.

## IMPLEMENTATION
Jalankan **Run All** dari lingkungan Python proyek. Keluaran aktif berada di folder `figures/` di samping notebook. Tiap run membuat laporan baru, tidak menimpa report lama.


In [ ]:
BENCHMARK = 'GSM8K'
NOTEBOOK_NAME = 'linguistic_marker_religion_gsm8k_bab4_visual_v1_3_1_volcano.ipynb'

from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
import dataclasses, hashlib, itertools, json, math, os, re, shutil, sys, unicodedata, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, Patch
from scipy.stats import rankdata, wilcoxon, binomtest
from IPython.display import Image, display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'AGENTS.md').is_file() and (p/'result').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from src.core.linguistic_markers.registry import MarkerSpec, PatternSpec
from src.core.linguistic_markers.extractor import extract_markers
from src.benchmarks.ethics.religion_lm_v3_extract import extract_final
from src.benchmarks.ethics.gender_lm_v2.surfaces import build_surfaces
ACTIVE = ROOT/'notebooks/religion/linguistic marker'/BENCHMARK
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
RUN = ROOT/'result/linguistic_markers'/f'religion_{BENCHMARK.lower()}_raw_rebuild_{RUN_ID}'
RUN.mkdir(parents=True,exist_ok=False)
FIGURES = RUN/'figures'
FIGURES.mkdir()
SEED = 20261005
LOW_SUPPORT = 20
SCIENTIFIC_STATUS = 'candidate_pending_independent_human_audit'
MODELS = ['meta-llama_llama-3.1-8b-instruct','meta-llama_llama-3.3-70b-instruct','mistralai_ministral-8b-2512','deepseek_deepseek-r1-distill-qwen-32b','openai_gpt-oss-20b','qwen_qwen3-next-80b-a3b-thinking']
MODEL_LABELS = dict(zip(MODELS,['Llama 3.1 8B','Llama 3.3 70B','Ministral 8B','DeepSeek R1 32B','GPT-OSS 20B','Qwen3 Next 80B']))
COLORS = dict(zip(MODELS,['#2463EB','#8038DC','#00946A','#D52C37','#C26A00','#007E99']))
IDENTITIES = ['islam','christian','jewish','atheist']
IDENTITY_LABELS = dict(zip(IDENTITIES,['Islam','Kristen','Yahudi','Ateis']))
CONDITIONS = ['explicit','implicit_1','implicit_2','implicit_3']
CL = dict(zip(CONDITIONS,['E','I1','I2','I3']))
PERSONAS = ['neutral']+[f'{i}_{c}' for i in IDENTITIES for c in CONDITIONS]
PL = {'neutral':'N', **{f'{i}_{c}':f'{IDENTITY_LABELS[i]} {CL[c]}' for i in IDENTITIES for c in CONDITIONS}}
OLD_G = ROOT/'result/GMS8K/reports/Religion_Persona/linguistic_markers/religion_gsm8k_full_source_aligned_v1_3_20260818'
OLD_E = ROOT/'result/ETHICS/reports/religion_lm_v3_candidate_20260916_r4_overall'
def sha(s):
    return hashlib.sha256(s.encode('utf-8')).hexdigest()
def normalized(s):
    return re.sub(r'\s+',' ',unicodedata.normalize('NFC',s or '')).strip()
def save_table(frame,name):
    frame.to_csv(RUN/name,index=False,encoding='utf-8-sig')
def file_sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
if BENCHMARK == 'GSM8K':
    registry_path = OLD_G/'marker_registry.csv'
    REGISTRY = pd.read_csv(registry_path,keep_default_na=False).to_dict('records')
    for m in REGISTRY:
        m['patterns']=json.loads(m['patterns'])
        m['benchmarks']=m['benchmarks'].split(';')
else:
    registry_path = OLD_E/'registry/markers.json'
    REGISTRY = json.loads(registry_path.read_text(encoding='utf-8'))
def make_marker(m):
    fields={f.name for f in dataclasses.fields(MarkerSpec)}
    d={k:v for k,v in m.items() if k in fields}
    d['patterns']=tuple(PatternSpec(**p) for p in d['patterns'])
    d['benchmarks']=tuple(d['benchmarks'])
    return MarkerSpec(**d)
MARKERS = tuple(make_marker(m) for m in REGISTRY)
MARKER_IDS = [m.marker_id for m in MARKERS]
REGISTRY_BY_ID = {m.marker_id:m for m in MARKERS}
(RUN/'registry.json').write_text(json.dumps(REGISTRY,ensure_ascii=False,indent=2),encoding='utf-8')
label_registry = pd.read_csv(ROOT/'result/ETHICS/reports/religion_ethics_gsm8k_style_20260918_r2/tables/registry.csv')
LABELS = dict(zip(label_registry.marker_id,label_registry.label))
LABELS.update({
 'organization_section_header':'Subjudul','organization_explicit_step':'Langkah eksplisit',
 'response_stance_committed':'Sikap evaluatif tegas','response_stance_conditional':'Sikap evaluatif bersyarat',
 'response_stance_refusal':'Penolakan evaluasi','response_binary_decline':'Penolakan jawaban biner',
 'word_count':'Jumlah kata','avg_sentence_length':'Kata per kalimat','ttr_300':'TTR hingga 300 kata',
 'math_latex_display':'Rumus LaTeX terpisah','math_inline_arithmetic':'Aritmetika dalam teks',
 'math_boxed_answer':'Jawaban dalam kotak','religion_scripture_authority':'Rujukan kitab suci',
 'theme_harm':'Tema bahaya / kerugian','theme_compassion':'Tema kepedulian',
 'theme_fairness':'Tema keadilan','theme_consent_autonomy':'Tema persetujuan / otonomi',
 'theme_boundaries':'Tema batas pribadi','theme_responsibility':'Tema tanggung jawab',
 'theme_intention':'Tema niat','theme_honesty':'Tema kejujuran'})
def label(m):
    return LABELS.get(m,m.replace('_',' '))
CONTRASTS=[]
for i in IDENTITIES:
    for c in CONDITIONS:
        CONTRASTS.append(dict(left=f'{i}_{c}',right='neutral',role='vs_neutral',group=i,condition=c))
    for c in CONDITIONS[1:]:
        CONTRASTS.append(dict(left=f'{i}_{c}',right=f'{i}_explicit',role='vs_explicit',group=i,condition=c))
for a,b in itertools.combinations(IDENTITIES,2):
    for c in CONDITIONS:
        CONTRASTS.append(dict(left=f'{a}_{c}',right=f'{b}_{c}',role='direct',group=f'{a}_minus_{b}',condition=c))
assert len(CONTRASTS)==52
DATASET_PATH = ROOT/('data/gsm8k_test.json' if BENCHMARK=='GSM8K' else 'data/ETHICS_test_4subsets_10percent_from200.json')
dataset = json.loads(DATASET_PATH.read_text(encoding='utf-8'))
canonical_by_id={}
canonical_by_hash=defaultdict(list)
for i,r in enumerate(dataset):
    subset='gsm8k' if BENCHMARK=='GSM8K' else r['subset']
    q=r['question'] if BENCHMARK=='GSM8K' else r['scenario']
    key=f'gsm8k_test_{i:05d}' if BENCHMARK=='GSM8K' else r['id']
    h=sha(normalized(q))
    canonical_by_id[(subset,key)]=h
    canonical_by_hash[(subset,h)].append(key)
SUBSETS=['gsm8k'] if BENCHMARK=='GSM8K' else ['commonsense','deontology','justice','virtue']
SCOPES=SUBSETS if BENCHMARK=='GSM8K' else SUBSETS+['gabungan']
SCOPE_LABELS={'gsm8k':'GSM8K','commonsense':'Commonsense','deontology':'Deontology','justice':'Justice','virtue':'Virtue','gabungan':'Gabungan empat subset'}
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':11,'axes.titlesize':13,'axes.labelsize':11,'xtick.labelsize':10,'ytick.labelsize':10,'savefig.dpi':160})
SOURCE_HASHES={str(p.relative_to(ROOT)):file_sha(p) for p in [registry_path,DATASET_PATH]}
for folder in [ROOT/'src/core/linguistic_markers',ROOT/'src/benchmarks/ethics']:
    paths=folder.glob('*.py') if folder.name=='linguistic_markers' else folder.glob('religion_lm_v3*.py')
    for p in paths:
        SOURCE_HASHES[str(p.relative_to(ROOT))]=file_sha(p)
for p in [ROOT/'src/benchmarks/ethics/gender_lm_v2/surfaces.py']:
    SOURCE_HASHES[str(p.relative_to(ROOT))]=file_sha(p)
print(BENCHMARK, '| seluruh data:',len(dataset),'soal x 6 model x 17 skenario')
print('Output baru:',RUN)


In [ ]:
def final_channel(obj):
    response=obj.get('response') or {}
    raw=response.get('raw_api_response') or {}
    choices=raw.get('choices') if isinstance(raw,dict) else None
    message=choices[0].get('message',{}) if choices and isinstance(choices[0],dict) else {}
    message=message if isinstance(message,dict) else {}
    reasoning=message.get('reasoning') or ''
    reasoning=reasoning if isinstance(reasoning,str) else ''
    if 'content' in message:
        value,source=message['content'],'api_message_content'
    elif 'content' in response:
        value,source=response['content'],'response_content_without_api_content'
    else:
        value,source='','no_final_content_field'
    status='ok'
    if value is not None and not isinstance(value,str):
        value,status='','unsupported_content_type'
    text=value or ''
    if response.get('success') is False:
        text,status='','api_failed'
    return text,source,status,len(reasoning),sha(normalized(reasoning)) if reasoning else ''

def canonical_key(subset,original,qhash):
    if not qhash:
        return '', 'missing_question'
    if canonical_by_id.get((subset,original))==qhash:
        return original,'id_and_content_verified'
    candidates=canonical_by_hash.get((subset,qhash),[])
    if len(candidates)==1:
        return candidates[0],'unique_content_match'
    return '', 'ambiguous_content' if candidates else 'outside_dataset'

def measure(content):
    if BENCHMARK=='GSM8K':
        surface=build_surfaces(content)
        visible='' if surface.status=='unknown_surface' else surface.visible_text
        values=extract_markers({'benchmark':'gsm8k','response_content':visible},MARKERS)
        values.pop('response_content',None)
        values.pop('benchmark',None)
        if re.fullmatch(r'[0-9]+',visible.strip()):
            values.update(text_available=False,text_status='answer_only_numeric')
        if surface.status=='unknown_surface':
            values.update(text_available=False,text_status='unknown_surface')
        values['explanation_available']=bool(values['text_available'])
        values['surface_flags']=';'.join(surface.flags)
        values.pop('visible_text',None)
        values.pop('explanation_text',None)
    else:
        values,_=extract_final(content,MARKERS)
        values['text_status']=values['surface_status']
        values['avg_sentence_length']=values['word_count']/values['sentence_count'] if values['explanation_available'] and values['sentence_count'] else None
    available=bool(values['explanation_available'])
    if not available:
        for col in ['word_count','sentence_count','avg_sentence_length','ttr_300']:
            values[col]=None
        for m in MARKER_IDS:
            for suffix in ['count','density','present']:
                values[f'{m}__{suffix}']=None
    return values

assert final_channel({'response':{'content':'WRAPPER','raw_api_response':{'choices':[{'message':{'content':None,'reasoning':'SECRET'}}]}}})[0]==''
assert final_channel({'response':{'success':False,'content':'partial'}})[0]==''
assert final_channel({'response':{'content':'visible'}})[0]=='visible'
assert measure('')['word_count'] is None
assert canonical_key('absent','x','y')[0]==''
assert bool(MARKERS[3].compiled_patterns[0].search('maybe this works'))
if BENCHMARK=='GSM8K':
    assert measure('6')['explanation_available'] is False
    assert measure('Maybe this works because it is likely.')['stance_hedge__count']==2

# Cache reuse is permitted only for the same frozen instrument and same final text.
CACHE={}
CACHE_COLUMNS=[]
if BENCHMARK=='ETHICS':
    from src.benchmarks.ethics.religion_lm_v3_registry import get_registry
    current={m.marker_id:dataclasses.asdict(m) for m in get_registry()}
    for m in MARKERS:
        old=dataclasses.asdict(m)
        for field in ['patterns','surface','measurement','exclusion_rule']:
            assert old[field]==current[m.marker_id][field],(m.marker_id,field)
    cache_path=OLD_E/'response_level/markers.csv.gz'
    SOURCE_HASHES[str(cache_path.relative_to(ROOT))]=file_sha(cache_path)
    CACHE_COLUMNS=['raw_final_sha256','surface_status','explanation_available','word_count','sentence_count','ttr_300','surface_flags']
    CACHE_COLUMNS += [f'{m}__{s}' for m in MARKER_IDS for s in ['count','density','present']]
    cached=pd.read_csv(cache_path,usecols=['model_id','persona','subset','question_id']+CACHE_COLUMNS,keep_default_na=False,float_precision='round_trip')
    CACHE={tuple(r[:4]):dict(zip(CACHE_COLUMNS,r[4:])) for r in cached[['model_id','persona','subset','question_id']+CACHE_COLUMNS].itertuples(index=False,name=None)}
    assert len(CACHE)==len(cached)
    del cached
    print('Pengukuran ETHICS lama dimuat untuk verifikasi final per respons:',len(CACHE))

def raw_paths(model,persona):
    if BENCHMARK=='GSM8K':
        domain='Gender_Persona' if persona=='neutral' else 'Religion_Persona'
        return sorted((ROOT/'result/GMS8K'/domain/model/persona).glob('gsm8k_*.json'))
    base=ROOT/'result/ETHICS/v9 [THIS IS THE MAIN]'
    if persona!='neutral':
        return sorted((base/'Religion Persona'/model/persona).glob('ethics_*.json'))
    folder=base/'Neutral Persona'/model
    return sorted((folder/'neutral').glob('ethics_*.json')) if (folder/'neutral').is_dir() else sorted(folder.glob('ethics_*.json'))

def read_record(task):
    model,persona,path=task
    binary=path.read_bytes()
    error=''
    try:
        obj=json.loads(binary)
        if not isinstance(obj,dict):
            raise ValueError('JSON must be an object')
    except (ValueError,UnicodeError) as exc:
        obj={}
        error=type(exc).__name__+': '+str(exc)
    subset='gsm8k' if BENCHMARK=='GSM8K' else obj.get('subset',path.stem.split('_')[1])
    assert subset in SUBSETS,(path,subset)
    q=(obj.get('input') or {}).get('question','') if BENCHMARK=='GSM8K' else obj.get('scenario','')
    q=q if isinstance(q,str) else ''
    qhash=sha(normalized(q)) if normalized(q) else ''
    key,pair_status=canonical_key(subset,path.stem,qhash)
    request=obj.get('request') or {}
    messages=request.get('messages') or []
    users=[m.get('content','') for m in messages if isinstance(m,dict) and m.get('role')=='user']
    last_user=users[-1] if users and isinstance(users[-1],str) else ''
    request_verified=bool(q) and normalized(q) in normalized(last_user)
    content,source,status,reasoning_chars,reasoning_hash=final_channel(obj)
    previous=CACHE.get((model,persona,subset,path.stem))
    verified_cache=previous is not None and previous['raw_final_sha256'] in (sha(content),sha(content.strip()))
    if verified_cache:
        values=previous.copy()
        values['text_status']=values['surface_status']
        values['avg_sentence_length']=float(values['word_count'])/float(values['sentence_count']) if values['explanation_available'] and values['sentence_count'] else None
        for col in CACHE_COLUMNS:
            if '__' in col or col in ['word_count','sentence_count','ttr_300']:
                values[col]=float(values[col]) if values[col]!='' else None
        if not values['explanation_available']:
            for col in ['word_count','sentence_count','avg_sentence_length','ttr_300']+[f'{m}__{s}' for m in MARKER_IDS for s in ['count','density','present']]:
                values[col]=None
    else:
        values=measure(content)
    if error:
        source,status='invalid_json','invalid_json'
    history=[m.get('content','') for m in messages if isinstance(m,dict) and m.get('role')=='assistant' and isinstance(m.get('content'),str)]
    values.update(model_id=model,persona=persona,subset=subset,original_question_id=path.stem,question_id=key,
        question_hash=qhash,question=q,canonical_status=pair_status,request_question_verified=request_verified,
        raw_source_file=str(path.relative_to(ROOT)),raw_sha256=hashlib.sha256(binary).hexdigest(),
        final_text=content,final_text_hash=sha(content),final_content_chars=len(content),
        text_source=source,source_status=status,source_error=error,reasoning_chars=reasoning_chars,reasoning_hash=reasoning_hash,
        reasoning_used=False,measurement_source='verified_frozen_final_cache' if verified_cache else 'recomputed_final_only',
        assistant_history_turns=len(history),assistant_history_words=sum(len(t.split()) for t in history),
        scientific_status=SCIENTIFIC_STATUS)
    return values


In [ ]:
tasks=[(model,persona,path) for model in MODELS for persona in PERSONAS for path in raw_paths(model,persona)]
assert tasks
assert all(raw_paths(model,persona) for model in MODELS for persona in PERSONAS)
print('Membaca seluruh JSON:',len(tasks),flush=True)
rows=[]
# Threads avoid copying the verified ETHICS measurement cache into worker processes.
with ThreadPoolExecutor(max_workers=6) as pool:
    for i,row in enumerate(pool.map(read_record,tasks),1):
        rows.append(row)
        if i%2000==0 or i==len(tasks):
            print(f'{BENCHMARK}: {i:,}/{len(tasks):,}',flush=True)
records=pd.DataFrame(rows)
del rows
keys=['model_id','persona','subset','question_id']
records['duplicate_pair_key']=records.question_id.ne('') & records.duplicated(keys,keep=False)
records['pair_eligible']=records.question_id.ne('') & ~records.duplicate_pair_key & records.request_question_verified
records['reasoning_only']=records.final_content_chars.eq(0) & records.reasoning_chars.gt(0)
assert not records.reasoning_used.any()
assert not records.loc[records.reasoning_only,'explanation_available'].any()
assert records.loc[~records.explanation_available,'word_count'].isna().all()
assert not records.loc[records.pair_eligible].duplicated(keys).any()
for m in MARKER_IDS:
    available=records.explanation_available
    count=pd.to_numeric(records[f'{m}__count'])
    density=pd.to_numeric(records[f'{m}__density'])
    present=pd.to_numeric(records[f'{m}__present'])
    assert np.allclose(density[available],count[available]/records.loc[available,'word_count']*100,equal_nan=True),m
    assert np.array_equal(present[available],(count[available]>0).astype(int)),m
    assert density[~available].isna().all(),m
save_table(records,'response_records.csv.gz')
SOURCE_HASHES['response_records_sha256']=file_sha(RUN/'response_records.csv.gz')
audit_cols=['model_id','persona','subset','original_question_id','question_id','question_hash','canonical_status','request_question_verified',
'duplicate_pair_key','pair_eligible','raw_source_file','raw_sha256','source_status','source_error','text_source',
'final_content_chars','reasoning_chars','reasoning_only','explanation_available','measurement_source','assistant_history_turns','assistant_history_words']
save_table(records[audit_cols],'source_audit.csv')
coverage=records.groupby(['model_id','subset','persona'],as_index=False).agg(
 total=('raw_source_file','size'),available=('explanation_available','sum'),pair_eligible=('pair_eligible','sum'),
 reasoning_only=('reasoning_only','sum'),invalid_json=('source_status',lambda s:int(s.eq('invalid_json').sum())))
expected={s:sum(1 for subset,key in canonical_by_id if subset==s) for s in SUBSETS}
coverage['expected_questions']=coverage.subset.map(expected)
coverage['available_pct']=100*coverage.available/coverage.total
coverage['missing_raw_files']=(coverage.expected_questions-coverage.total).clip(lower=0)
save_table(coverage,'coverage.csv')
save_table(records.groupby(['model_id','persona','subset','text_status'],as_index=False).size(),'text_status_counts.csv')
save_table(records.groupby(['model_id','persona','subset','canonical_status','request_question_verified'],as_index=False).size(),'canonical_status_counts.csv')
save_table(records.groupby(['model_id','persona'],as_index=False).agg(
 total=('persona','size'),with_assistant_history=('assistant_history_turns',lambda x:int((x>0).sum())),
 median_history_words=('assistant_history_words','median'),max_history_words=('assistant_history_words','max')),'conversation_context.csv')
pair_audit=[]
for (model,subset),g in records.groupby(['model_id','subset'],sort=False):
    for c in CONTRASTS:
        a=g[g.persona.eq(c['left'])]
        b=g[g.persona.eq(c['right'])]
        old=a[['original_question_id','question_hash']].merge(b[['original_question_id','question_hash']],on='original_question_id',suffixes=('_left','_right'),validate='one_to_one')
        new=a[a.pair_eligible][['question_id','question_hash','explanation_available']].merge(
            b[b.pair_eligible][['question_id','question_hash','explanation_available']],on='question_id',suffixes=('_left','_right'),validate='one_to_one')
        assert new.question_hash_left.eq(new.question_hash_right).all()
        pair_audit.append(dict(model_id=model,subset=subset,**c,old_id_pairs=len(old),
            old_id_question_mismatch=int((old.question_hash_left.ne('')&old.question_hash_right.ne('')&old.question_hash_left.ne(old.question_hash_right)).sum()),
            old_id_question_unknown=int((old.question_hash_left.eq('')|old.question_hash_right.eq('')).sum()),
            verified_question_pairs=len(new),paired_explanations=int((new.explanation_available_left&new.explanation_available_right).sum())))
pair_audit=pd.DataFrame(pair_audit)
save_table(pair_audit,'pairing_audit.csv')
# Check cached measurement reproducibility on ten available rows per model.
cache_checks=[]
if BENCHMARK=='ETHICS':
    for model in MODELS:
        sample=records[records.model_id.eq(model)&records.explanation_available].sample(n=10,random_state=SEED)
        for _,r in sample.iterrows():
            fresh=measure(r.final_text)
            changed=[m for m in MARKER_IDS if fresh[f'{m}__count']!=r[f'{m}__count']]
            cache_checks.append(dict(model_id=model,raw_source_file=r.raw_source_file,marker_differences=';'.join(changed),word_count_matches=fresh['word_count']==r.word_count))
    save_table(pd.DataFrame(cache_checks),'cache_reproduction_checks.csv')
    assert all(not r['marker_differences'] and r['word_count_matches'] for r in cache_checks),'Cached instrument does not reproduce; stop before publishing.'
else:
    previous=pd.read_csv(OLD_G/'response_records.csv',usecols=['model_id','persona','question_id','visible_text'],keep_default_na=False)
    previous=previous.rename(columns={'question_id':'original_question_id'})
    previous['old_visible_hash']=previous.visible_text.map(lambda t:sha(normalized(t)) if normalized(t) else '')
    comparison=previous.drop(columns='visible_text').merge(records[['model_id','persona','original_question_id','final_content_chars','reasoning_hash','explanation_available']],on=['model_id','persona','original_question_id'],how='outer',indicator=True,validate='one_to_one')
    comparison['old_text_equals_reasoning']=comparison.old_visible_hash.ne('') & comparison.old_visible_hash.eq(comparison.reasoning_hash)
    save_table(comparison,'previous_source_comparison.csv')
(RUN/'source_checkpoint.json').write_text(json.dumps({'benchmark':BENCHMARK,'rows':len(records),'status':SCIENTIFIC_STATUS,'source_hashes':SOURCE_HASHES},indent=2),encoding='utf-8')
print('Pemeriksaan pengukuran dan pairing lolos. Ketidakcocokan nomor lama:',int(pair_audit.old_id_question_mismatch.sum()))
display(coverage.groupby('model_id')[['total','available','reasoning_only','invalid_json']].sum())


## CHECKLIST

- [x] Final-only, kamus beku, pairing isi soal, semua respons diinventarisasi.
- [x] Count/density/presence, dukungan pasangan, NA terpisah, I–E langsung.
- [x] Uji regresi kanal, exact signed-rank, Holm dan validasi PNG terpasang.
- [ ] Audit manusia independen kamus/parsing: belum selesai; output tetap kandidat.

## TESTING
Sel berikut menghitung efek dan menguji invariannya. k<20 adalah tanda dukungan rendah, bukan ambang validitas universal. TP tidak diubah menjadi RRBC nol. CI hanya untuk perubahan rata-rata karakteristik, bukan untuk RRBC. Semua p yang direncanakan termasuk dalam satu keluarga Holm per model untuk benchmark ini.


In [ ]:
def holm(pvalues):
    p=np.asarray(pvalues,float)
    out=np.full(p.shape,np.nan)
    idx=np.flatnonzero(np.isfinite(p))
    order=idx[np.argsort(p[idx],kind='stable')]
    out[order]=np.minimum(1,np.maximum.accumulate(p[order]*(len(p)-np.arange(len(order)))))
    return out

def signed_rank(delta):
    d=np.asarray(delta,float)
    d=d[np.isfinite(d)&(d!=0)]
    k=len(d)
    if not k:
        return np.nan,1.,'all_ties'
    ranks=rankdata(np.abs(d),method='average')
    effect=float((ranks[d>0].sum()-ranks[d<0].sum())/ranks.sum())
    if k<=20:
        weights=np.rint(ranks*2).astype(int)
        total=int(weights.sum())
        counts=np.zeros(total+1,dtype=np.int64)
        counts[0]=1
        for w in weights:
            old=counts.copy()
            counts[w:]+=old[:-w]
        observed=int(weights[d>0].sum())
        extreme=np.abs(2*np.arange(total+1)-total)>=abs(2*observed-total)
        p=float(counts[extreme].sum()/2**k)
        return effect,p,'signed_rank_exact_sign_permutation'
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',UserWarning)
        p=float(wilcoxon(d,zero_method='wilcox',alternative='two-sided',method='approx',correction=False).pvalue)
    return effect,p,'signed_rank_normal_approximation'

assert np.allclose(holm([.01,.04,.03]),[.03,.06,.06])
assert np.isnan(signed_rank([0,0])[0])
assert signed_rank([0,1,1])[0]==1
assert np.isclose(signed_rank([1]*20)[1],2/2**20)
for delta in [[1,-1,2],[1,2,-3,4],[-1,-1,1,2,0]]:
    d=np.array([x for x in delta if x])
    ranks=rankdata(abs(d))
    obs=abs(np.sum(np.sign(d)*ranks))
    brute=np.mean([abs(np.sum(np.array(s)*ranks))>=obs for s in itertools.product([-1,1],repeat=len(d))])
    assert np.isclose(brute,signed_rank(d)[1])

# One declared measurement per marker, plus presence sensitivity for density-based religion markers.
OUTCOMES=[dict(outcome=m.marker_id,marker_id=m.marker_id,column=f'{m.marker_id}__'+('present' if m.measurement=='binary' else 'density'),
 metric='presence' if m.measurement=='binary' else 'density',analysis='registered') for m in MARKERS]
OUTCOMES += [dict(outcome=m.marker_id+'__presence',marker_id=m.marker_id,column=f'{m.marker_id}__present',metric='presence',analysis='religion_presence_sensitivity')
 for m in MARKERS if m.marker_id.startswith('religion_') and m.measurement!='binary']
OUTCOMES += [dict(outcome=m,marker_id=m,column=m,metric='surface',analysis='surface') for m in ['word_count','avg_sentence_length','ttr_300']]
outcome_registry=pd.DataFrame(OUTCOMES)
save_table(outcome_registry,'outcome_registry.csv')
outcome_cols=outcome_registry.column.tolist()
stats=[]
support=[]
pooled_direction=[]
rng=np.random.default_rng(SEED)
total_jobs=len(MODELS)*len(SCOPES)*len(CONTRASTS)
done=0
for scope in SCOPES:
    for model in MODELS:
        frame=records[records.model_id.eq(model)&records.pair_eligible]
        if scope!='gabungan':
            frame=frame[frame.subset.eq(scope)]
        indexed={p:g.set_index(['subset','question_id']) for p,g in frame.groupby('persona')}
        for c in CONTRASTS:
            left=indexed[c['left']]
            right=indexed[c['right']]
            common=left.index.intersection(right.index)
            a=left.loc[common]
            b=right.loc[common]
            assert a.question_hash.eq(b.question_hash).all()
            ok=a.explanation_available.to_numpy() & b.explanation_available.to_numpy()
            a=a.loc[ok]
            b=b.loc[ok]
            pair_subsets=a.index.get_level_values('subset').to_numpy()
            n_by_subset={s:int(np.sum(pair_subsets==s)) for s in SUBSETS if scope=='gabungan' or s==scope}
            n=len(a)
            support.append(dict(model_id=model,scope=scope,**c,question_pairs=len(common),explanation_pairs=n,
                explanation_pair_pct=100*n/len(common) if len(common) else np.nan,
                subset_counts=json.dumps(n_by_subset),subsets_present=sum(v>0 for v in n_by_subset.values())))
            A=a[outcome_cols].to_numpy(float)
            B=b[outcome_cols].to_numpy(float)
            D=A-B
            # CI is only for paired mean changes in the three surface measurements.
            # Stratification fixes observed subset weights; it does not equalize them.
            surface_ci={}
            for j,out in enumerate(OUTCOMES):
                if out['metric']!='surface':
                    continue
                d=D[:,j]
                finite=np.isfinite(d)
                d=d[finite]
                ss=pair_subsets[finite]
                k=int(np.count_nonzero(d))
                ci=(np.nan,np.nan)
                ci_status='insufficient_support'
                if len(d)>=2 and k>=6 and np.ptp(d)>0 and all(np.sum(ss==s)>=2 for s in np.unique(ss)):
                    estimates=np.zeros(2000)
                    for s in np.unique(ss):
                        group=d[ss==s]
                        for start in range(0,2000,100):
                            draws=rng.integers(0,len(group),size=(min(100,2000-start),len(group)))
                            estimates[start:start+len(draws)]+=group[draws].sum(axis=1)/len(d)
                    ci=tuple(np.quantile(estimates,[.025,.975]))
                    ci_status='ok' if ci[0]!=ci[1] else 'degenerate_hidden'
                    if ci_status!='ok':
                        ci=(np.nan,np.nan)
                surface_ci[out['outcome']]=(*ci,ci_status)
            for j,out in enumerate(OUTCOMES):
                good=np.isfinite(D[:,j])
                d=D[good,j]
                n_out=len(d)
                k=int(np.count_nonzero(d))
                positive=int(np.sum(d>0))
                negative=int(np.sum(d<0))
                effect,p,method=signed_rank(d) if n_out else (np.nan,np.nan,'no_pairs')
                sign_p=binomtest(positive,k,.5).pvalue if k else (1. if n_out else np.nan)
                if out['metric']=='presence':
                    effect=float(np.mean(d)*100) if n_out else np.nan
                    p=sign_p
                    method='mcnemar_exact_conditional'
                ci_low,ci_high,ci_status=surface_ci.get(out['outcome'],(np.nan,np.nan,'not_computed_for_marker'))
                stats.append(dict(model_id=model,scope=scope,**c,**out,
                    n_pairs=n_out,n_nonzero=k,n_ties=n_out-k,n_positive=positive,n_negative=negative,
                    effect=effect,mean_delta=float(np.mean(d)) if n_out else np.nan,
                    median_delta=float(np.median(d)) if n_out else np.nan,p_raw=p,p_sign_sensitivity=sign_p,
                    test_method=method,mean_delta_ci_low=ci_low,mean_delta_ci_high=ci_high,ci_status=ci_status,
                    support_state='ND' if not n_out else ('TP' if not k else ('low' if k<LOW_SUPPORT else 'supported')),
                    low_support=bool(k<LOW_SUPPORT),scientific_status=SCIENTIFIC_STATUS))
            done+=1
            if done%52==0:
                print(f'Statistik: {done}/{total_jobs} kontras-scope',flush=True)
statistics=pd.DataFrame(stats)
pair_support=pd.DataFrame(support)
family_size=len(SCOPES)*len(CONTRASTS)*len(OUTCOMES)
statistics['holm_family_size']=family_size
statistics['holm_family']='all_planned_outcomes_contrasts_scopes_per_model'
statistics['p_holm']=np.nan
statistics['p_sign_holm']=np.nan
for model,idx in statistics.groupby('model_id').groups.items():
    assert len(idx)==family_size
    statistics.loc[idx,'p_holm']=holm(statistics.loc[idx,'p_raw'])
    statistics.loc[idx,'p_sign_holm']=holm(statistics.loc[idx,'p_sign_sensitivity'])
statistics['significant']=statistics.p_holm.lt(.05)
assert statistics.loc[statistics.support_state.eq('TP')&statistics.metric.ne('presence'),'effect'].isna().all()
assert statistics.loc[statistics.support_state.eq('ND'),'p_raw'].isna().all()
save_table(statistics,'paired_statistics.csv.gz')
save_table(pair_support,'pair_support.csv')
if BENCHMARK=='ETHICS':
    merge_cols=['model_id','left','right','outcome']
    subset_stats=statistics[statistics.scope.ne('gabungan')][merge_cols+['scope','effect','n_pairs','n_nonzero']].copy()
    overall_stats=statistics[statistics.scope.eq('gabungan')][merge_cols+['effect']].rename(columns={'effect':'pooled_effect'})
    direction=subset_stats.merge(overall_stats,on=merge_cols,validate='many_to_one')
    direction['same_nonzero_direction']=np.where(direction.effect.notna()&direction.pooled_effect.notna()&direction.effect.ne(0)&direction.pooled_effect.ne(0),
         np.sign(direction.effect)==np.sign(direction.pooled_effect),np.nan)
    save_table(direction,'pooled_subset_direction_check.csv')
print('Statistik selesai:',len(statistics),'baris; Holm',family_size,'hipotesis per model.')


In [ ]:
figure_index=[]
def save_figure(fig,name,kind,scope,detail=''):
    fig.savefig(FIGURES/name,dpi=160,facecolor='white')
    plt.close(fig)
    figure_index.append(dict(file=name,kind=kind,scope=scope,detail=detail,status=SCIENTIFIC_STATUS))
    if len(figure_index)%20==0:
        print('PNG selesai:',len(figure_index),flush=True)

def group_title(role,group):
    if role=='direct':
        a,b=group.split('_minus_')
        return IDENTITY_LABELS[a]+' − '+IDENTITY_LABELS[b]
    return IDENTITY_LABELS[group]+(' − Netral' if role=='vs_neutral' else ': implisit − eksplisit')

# Each family uses one metric. No mixed rank-biserial / percentage-point color scale.
COMMON=MARKER_IDS[:14]
RELIGION=[m for m in MARKER_IDS if m.startswith('religion_')]
CORE=[m for m in ['framing_deontological','framing_consequentialist','framing_virtue_ethics','framing_religious','framing_cultural'] if m in MARKER_IDS]
SUPPLEMENT=[m for m in ['stance_contextual_hedge','stance_distancing','connective_additive','connective_concessive','affect_empathy_validation','moral_evaluation_positive','moral_evaluation_negative'] if m in MARKER_IDS]
THEMES=[m for m in MARKER_IDS if m.startswith('theme_')]
families=[('bahasa_umum','Bahasa umum',COMMON,'density')]
for start in range(0,len(RELIGION),10):
    mids=RELIGION[start:start+10]
    outcomes=[m if REGISTRY_BY_ID[m].measurement=='binary' else m+'__presence' for m in mids]
    families.append((f'penanda_agama_{start//10+1}','Penanda agama · prevalensi',outcomes,'presence'))
if CORE:
    families += [('etika_inti','Penanda etika inti',CORE,'density'),
                 ('etika_tambahan','Penanda etika tambahan',SUPPLEMENT,'density'),
                 ('tema_etika','Tema etika · kamus kandidat',THEMES,'density')]
family_by_outcome={o:f[0] for f in families for o in f[2]}

for scope in SCOPES:
    cov=coverage[coverage.subset.eq(scope)].copy() if scope!='gabungan' else coverage.groupby(['model_id','persona'],as_index=False)[['total','available','expected_questions']].sum()
    cov['available_pct']=100*cov.available/cov.total
    fig,ax=plt.subplots(figsize=(19,5.5))
    grid=cov.pivot(index='model_id',columns='persona',values='available_pct').reindex(index=MODELS,columns=PERSONAS)
    im=ax.imshow(grid.to_numpy(float),vmin=0,vmax=100,cmap='YlGnBu',aspect='auto')
    lookup=cov.set_index(['model_id','persona'])
    for i,m in enumerate(MODELS):
        for j,p in enumerate(PERSONAS):
            r=lookup.loc[(m,p)]
            color='white' if r.available_pct>55 else '#172033'
            ax.text(j,i,f'{r.available_pct:.1f}%\n{int(r.available)}/{int(r.total)}',ha='center',va='center',fontsize=8,color=color)
    ax.set_xticks(range(17),[PL[p] for p in PERSONAS],rotation=45,ha='right')
    ax.set_yticks(range(6),[MODEL_LABELS[m] for m in MODELS])
    ax.set_title(f'Ketersediaan penjelasan · {SCOPE_LABELS[scope]}',pad=15,fontweight='bold')
    fig.colorbar(im,ax=ax,label='Persentase respons dengan penjelasan')
    fig.text(.5,.02,'Kandidat · pembilang/penyebut = penjelasan tersedia/JSON ditemukan; kegagalan API dan final kosong tetap dicatat.',ha='center',fontsize=10)
    fig.subplots_adjust(left=.14,right=.96,bottom=.24,top=.89)
    save_figure(fig,f'00_cakupan_{scope}.png','coverage',scope)

    ps=pair_support[pair_support.scope.eq(scope)]
    fig,axes=plt.subplots(3,1,figsize=(22,12),layout='constrained')
    for ax,role in zip(axes,['vs_neutral','vs_explicit','direct']):
        part=ps[ps.role.eq(role)]
        contrasts=part[['left','right']].drop_duplicates().apply(tuple,axis=1).tolist()
        matrix=np.array([[int(part[part.model_id.eq(m)&part.left.eq(a)&part.right.eq(b)].explanation_pairs.iloc[0]) for a,b in contrasts] for m in MODELS])
        vmax=max(1,matrix.max())
        im=ax.imshow(matrix,cmap='Blues',vmin=0,vmax=vmax,aspect='auto')
        for i in range(6):
            for j in range(len(contrasts)):
                ax.text(j,i,str(matrix[i,j]),ha='center',va='center',fontsize=8,color='white' if matrix[i,j]>vmax*.55 else '#172033')
        ax.set_xticks(range(len(contrasts)),[PL[a]+' − '+PL[b] for a,b in contrasts],rotation=35,ha='right',fontsize=8)
        ax.set_yticks(range(6),[MODEL_LABELS[m] for m in MODELS],fontsize=9)
        ax.set_title({'vs_neutral':'Persona − Netral','vs_explicit':'Implisit − Eksplisit','direct':'Antaridentitas'}[role])
    fig.suptitle(f'Dukungan pasangan penjelasan · {SCOPE_LABELS[scope]}',fontweight='bold',fontsize=18)
    save_figure(fig,f'01_dukungan_pasangan_{scope}.png','paired_coverage',scope)

    for family_name,family_label,outcomes,metric in families:
        for role in ['vs_neutral','vs_explicit','direct']:
            # Core/additional ethics and themes retain all direct effects in CSV;
            # their primary figures focus on persona−N and I−E.
            if role=='direct' and family_name in ['etika_tambahan','tema_etika']:
                continue
            groups=list(dict.fromkeys(c['group'] for c in CONTRASTS if c['role']==role))
            for group in groups:
                condition_order=CONDITIONS[1:] if role=='vs_explicit' else CONDITIONS
                part=statistics[statistics.scope.eq(scope)&statistics.role.eq(role)&statistics.group.eq(group)&statistics.outcome.isin(outcomes)]
                fig,axes=plt.subplots(2,3,figsize=(23,max(10,3.8+len(outcomes)*.60)))
                limit=100 if metric=='presence' else 1
                cmap=plt.get_cmap('RdBu_r').copy()
                cmap.set_bad('#e9edf2')
                for ax,model in zip(axes.flat,MODELS):
                    table=part[part.model_id.eq(model)].set_index(['outcome','condition'])
                    values=np.array([[table.loc[(o,c),'effect'] for c in condition_order] for o in outcomes],float)
                    im=ax.imshow(values,cmap=cmap,vmin=-limit,vmax=limit,aspect='auto')
                    for y,o in enumerate(outcomes):
                        for x,c in enumerate(condition_order):
                            r=table.loc[(o,c)]
                            if r.support_state in ['ND','TP']:
                                value=r.support_state
                                ax.add_patch(Rectangle((x-.5,y-.5),1,1,facecolor='#e9edf2',edgecolor='white',linewidth=.5))
                            else:
                                value=f'{r.effect:+.1f}' if metric=='presence' else f'{r.effect:+.2f}'
                                value+=('*' if r.significant else '')+('†' if r.low_support else '')
                                if r.low_support:
                                    ax.add_patch(Rectangle((x-.5,y-.5),1,1,fill=False,hatch='///',edgecolor='#727b87',linewidth=0))
                            color='white' if pd.notna(r.effect) and abs(r.effect)>limit*.65 and r.support_state not in ['ND','TP'] else '#162031'
                            ax.text(x,y-.12,value,ha='center',va='center',fontsize=10,fontweight='bold',color=color)
                            ax.text(x,y+.22,f'k={r.n_nonzero} / n={r.n_pairs}',ha='center',va='center',fontsize=7.5,color=color)
                    ax.set_title(MODEL_LABELS[model],color=COLORS[model],fontweight='bold',pad=10)
                    ax.set_xticks(range(len(condition_order)),[CL[c]+('−E' if role=='vs_explicit' else '') for c in condition_order])
                    mids=[o.removesuffix('__presence') for o in outcomes]
                    ax.set_yticks(range(len(outcomes)),[label(m) for m in mids],fontsize=9)
                    ax.tick_params(length=0)
                    for spine in ax.spines.values():
                        spine.set_visible(False)
                title=f'{family_label}: {group_title(role,group)}'
                fig.suptitle(title+'\n'+SCOPE_LABELS[scope]+' · kandidat',fontsize=19,fontweight='bold',y=.985)
                fig.subplots_adjust(left=.14,right=.98,bottom=.18,top=.87,hspace=.35,wspace=.70)
                cax=fig.add_axes([.34,.105,.40,.018])
                cb=fig.colorbar(im,cax=cax,orientation='horizontal')
                cb.set_label('Selisih prevalensi (poin persentase), kiri − kanan' if metric=='presence' else 'Matched rank-biserial dari densitas per 100 kata, kiri − kanan',fontsize=10)
                fig.text(.5,.053,'* p Holm < 0,05 · arsiran/†: k < 20 (aturan penandaan dukungan rendah, bukan batas universal).',ha='center',fontsize=10)
                fig.text(.5,.029,'k = pasangan berubah; n = pasangan penjelasan · ND = tidak ada pasangan · TP = pasangan ada, semua selisih nol.',ha='center',fontsize=10)
                fig.text(.5,.010,f'Holm: {family_size:,} hipotesis per model, seluruh outcome/kontras/scope; status ND/TP tidak diberi warna efek kuat.',ha='center',fontsize=9,color='#475569')
                save_figure(fig,f'02_efek_{family_name}_{role}_{group}_{scope}.png','effect_heatmap',scope,title)

    # Paired surface changes: shared horizontal range across every scenario.
    for role in ['vs_neutral','vs_explicit']:
        groups=IDENTITIES
        fig,axes=plt.subplots(3,4,figsize=(22,12),squeeze=False)
        for row,outcome in enumerate(['word_count','avg_sentence_length','ttr_300']):
            data=statistics[statistics.scope.eq(scope)&statistics.role.eq(role)&statistics.outcome.eq(outcome)]
            finite=np.concatenate([data.mean_delta.dropna(),data.mean_delta_ci_low.dropna(),data.mean_delta_ci_high.dropna()])
            bound=max(abs(finite).max()*1.1, .01) if len(finite) else 1
            for col,group in enumerate(groups):
                ax=axes[row,col]
                conditions=CONDITIONS[1:] if role=='vs_explicit' else CONDITIONS
                for mi,model in enumerate(MODELS):
                    part=data[data.group.eq(group)&data.model_id.eq(model)]
                    for ci,c in enumerate(conditions):
                        r=part[part.condition.eq(c)].iloc[0]
                        y=ci+(mi-2.5)*.10
                        if pd.isna(r.mean_delta):
                            continue
                        ax.scatter(r.mean_delta,y,s=40,c=COLORS[model],marker='^' if r.low_support else 'o',
                            edgecolor='black' if r.significant else 'white',linewidth=1,zorder=4)
                        if pd.notna(r.mean_delta_ci_low):
                            ax.plot([r.mean_delta_ci_low,r.mean_delta_ci_high],[y,y],color=COLORS[model],linewidth=1.5)
                ax.axvline(0,color='#7b8798',lw=1)
                ax.set_xlim(-bound,bound)
                ax.set_yticks(range(len(conditions)),[CL[c]+('−E' if role=='vs_explicit' else '−N') for c in conditions])
                ax.invert_yaxis()
                ax.grid(axis='x',alpha=.25)
                ax.set_xlabel('Selisih rata-rata · '+label(outcome),fontsize=10)
                ax.set_title(IDENTITY_LABELS[group] if row==0 else '',fontweight='bold')
        fig.suptitle('Perubahan karakteristik penjelasan · '+SCOPE_LABELS[scope]+'\n'+('Persona − Netral' if role=='vs_neutral' else 'Implisit − Eksplisit')+' · kandidat',fontsize=19,fontweight='bold')
        handles=[Line2D([],[],marker='o',color=COLORS[m],linestyle='',label=MODEL_LABELS[m]) for m in MODELS]
        fig.legend(handles=handles,loc='lower center',bbox_to_anchor=(.5,.065),ncol=6,frameon=False)
        fig.text(.5,.035,'Garis = CI bootstrap 95% selisih rata-rata (2.000 resampel), bukan CI rank-biserial. Skala sama per ukuran.',ha='center',fontsize=10)
        fig.text(.5,.013,'Segitiga: <20 pasangan berubah · tepi hitam: p Holm <0,05 · CI disembunyikan bila n<2, k<6, degenerat, atau strata <2.',ha='center',fontsize=10)
        fig.subplots_adjust(left=.07,right=.99,top=.88,bottom=.17,hspace=.45,wspace=.22)
        save_figure(fig,f'03_karakteristik_{role}_{scope}.png','paired_characteristics',scope)

    # Descriptive pages cover every registered marker, including binary/format QC outcomes.
    frame=records[records.explanation_available]
    if scope!='gabungan':
        frame=frame[frame.subset.eq(scope)]
    for page,start in enumerate(range(0,len(MARKER_IDS),12),1):
        mids=MARKER_IDS[start:start+12]
        profile=frame.groupby(['model_id','persona'])[[f'{m}__present' for m in mids]].mean()*100
        fig,axes=plt.subplots(2,3,figsize=(24,13))
        for ax,model in zip(axes.flat,MODELS):
            matrix=profile.loc[model].reindex(PERSONAS).T
            im=ax.imshow(matrix,vmin=0,vmax=100,cmap='YlGnBu',aspect='auto')
            ax.set_xticks(range(17),[PL[p] for p in PERSONAS],rotation=70,ha='right',fontsize=8)
            ax.set_yticks(range(len(mids)),[label(m) for m in mids],fontsize=9)
            ax.set_title(MODEL_LABELS[model],fontweight='bold',color=COLORS[model])
        fig.suptitle(f'Profil prevalensi penanda · {SCOPE_LABELS[scope]} · halaman {page}\nDeskriptif pada penjelasan tersedia; bukan uji perbedaan · kandidat',fontsize=18,fontweight='bold')
        fig.subplots_adjust(left=.13,right=.98,bottom=.17,top=.88,hspace=.60,wspace=.65)
        cax=fig.add_axes([.35,.04,.35,.018])
        fig.colorbar(im,cax=cax,orientation='horizontal',label='Persentase penjelasan yang memuat penanda')
        save_figure(fig,f'04_profil_semua_penanda_{scope}_{page:02d}.png','descriptive_profile',scope)
save_table(pd.DataFrame(figure_index),'figure_index.csv')
print('Jumlah PNG:',len(figure_index))


## INTEGRATION
Sel terakhir memeriksa seluruh PNG, statistik, dan hash sumber; setelah lolos, gambar lama dipindahkan ke arsip yang bisa dipulihkan, lalu PNG baru dipasang di figures/. Tidak menyinkronkan LaTeX. README dan manifest mencatat keterbatasan dan lokasi CSV sumber.


In [ ]:
from PIL import Image as PILImage
for entry in figure_index:
    with PILImage.open(FIGURES/entry['file']) as im:
        im.verify()
assert len(list(FIGURES.glob('*.png')))==len(figure_index)
assert not list(FIGURES.glob('*.svg'))
assert statistics.groupby('model_id').size().eq(family_size).all()
assert pair_support.explanation_pairs.le(pair_support.question_pairs).all()
assert all(file_sha(ROOT/p)==h for p,h in SOURCE_HASHES.items() if p!='response_records_sha256')
summary={
 'benchmark':BENCHMARK,'run_id':RUN_ID,'status':SCIENTIFIC_STATUS,
 'rows':len(records),'raw_json_invalid':int(records.source_status.eq('invalid_json').sum()),
 'available_explanations':int(records.explanation_available.sum()),
 'pair_eligible_records':int(records.pair_eligible.sum()),
 'request_question_unverified':int((~records.request_question_verified).sum()),
 'duplicate_pair_key_records':int(records.duplicate_pair_key.sum()),
 'reasoning_only_records':int(records.reasoning_only.sum()),
 'original_id_question_mismatches_across_contrasts':int(pair_audit.old_id_question_mismatch.sum()),
 'figures':len(figure_index),'markers':len(MARKER_IDS),'outcomes':len(OUTCOMES),
 'holm_hypotheses_per_model':family_size,'source_hashes':SOURCE_HASHES,
 'marker_dictionary_changed':False,'answer_scoring_changed':False,'reasoning_used':False,
 'human_audit_passed':False,'notebook_source':NOTEBOOK_NAME,
 'numeric_test':'signed-rank exact conditional sign permutation k<=20; normal approximation otherwise',
 'binary_test':'exact conditional McNemar','bootstrap':'surface paired mean only, 2000, fixed observed subset weights',
 'scientific_limits':['Independent human marker/parser audit incomplete','Implicit condition changes dialogue history as well as identity expression',
  'Available-explanation selection differs by model/persona','Pooled ETHICS is pair-weighted, not equal-subset weighted',
  'GSM8K and ETHICS frozen dictionaries and tokenization are not identical; do not equate raw prevalence across benchmarks'],
}
(RUN/'manifest.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
notes=f"""# Linguistic marker religion — {BENCHMARK}
Tanggal: 2026-10-05. Status: **kandidat; audit manusia independen belum selesai**.

## Lokasi
Notebook aktif: {NOTEBOOK_NAME}. PNG aktif: figures/. Sumber lengkap keluaran ini: {RUN}.
Tidak ada HTML/SVG baru. Hasil lama disimpan di _arsip/, bukan dihapus.
Run All membaca seluruh korpus yang tersedia, membuat direktori hasil baru, lalu memasang gambar setelah pemeriksaan lolos.

## Yang berubah
- Pairing memakai isi soal yang cocok dengan dataset dan terverifikasi muncul dalam pesan pengguna terakhir. Nomor file saja tidak cukup.
- GSM8K memakai seluruh 1.319 soal test; ETHICS memakai seluruh 1.514 soal dalam eksperimen empat subset, bukan seluruh benchmark ETHICS global.
- API final kosong tidak dipulihkan dari reasoning. Semua respons tetap diinventarisasi, tanpa filter committed/correct.
- Kamus dibekukan: {len(MARKER_IDS)} marker; definisi/regex tidak ditambah. Setiap sumber, definisi, dan pengecualian ada di registry.json.
- Densitas dan prevalensi dipisahkan. Jumlah mentah, densitas per 100 kata, dan kehadiran disimpan per respons.
- Seluruh 52 kontras/model tersedia: 16 persona–N, 12 I–E, 24 antaridentitas. Arah selalu **kiri dikurangi kanan**, tertulis di judul.
- Warna tidak dipudarkan berdasarkan p-value. Heatmap mencantumkan k/n, arsiran dukungan rendah, dan tanda signifikansi.
- Keluarga Holm yang sama mencakup seluruh {family_size:,} hipotesis terencana/model dalam paket benchmark ini, termasuk subset dan gabungan untuk ETHICS. Hipotesis tidak teruji tetap dihitung dalam ukuran keluarga, p-nya tetap NA.

## Membaca gambar
- **ND**: tidak ada pasangan penjelasan yang dapat dibandingkan. Bukan efek nol.
- **TP**: pasangan ada tetapi semua selisih nol; n tetap terlihat. Rank-biserial tidak terdefinisi dan tidak diubah menjadi nol. Untuk prevalensi, selisih aktual nol tetap tersedia di CSV.
- **k**: pasangan dengan selisih nonnol; **n**: pasangan penjelasan yang dibandingkan.
- Arsiran/† atau segitiga: k<20, hanya aturan penandaan dukungan rendah, bukan batas universal validitas.
- Bintang atau tepi hitam: p Holm<0,05.
- Rank-biserial dihitung dari peringkat selisih densitas per respons. Nilai ±1 tidak berarti perubahan luas; selalu baca k/n.
- Prevalensi: selisih persentase respons dengan marker, dalam poin persentase; uji McNemar eksak.
- Garis CI pada karakteristik adalah CI **selisih rata-rata**, bukan CI rank-biserial. Bootstrap 2.000 kali pada pasangan; ETHICS gabungan distratifikasi subset dengan bobot pasangan tersedia. CI disembunyikan bila n<2, k<6, degenerat, atau strata berisi <2 pasangan.
- Profil semua marker bersifat deskriptif. Tidak menyiratkan signifikansi; semua efek inferensial tetap ada di CSV.

## Kebijakan statistik dan perbandingan dengan paket lama
Ini bukan sekadar perubahan tata letak. Pasangan soal dan kanal teks diperbaiki.
Uji numerik kini diselaraskan dengan paket gender: signed-rank dua sisi, exact sign permutation pada k≤20 (mendukung ties), aproksimasi normal pada k>20.
Uji tanda disimpan sebagai sensitivitas dalam p_sign_sensitivity dan p_sign_holm.
Paket GSM8K lama memilih sign/Wilcoxon secara adaptif dan keluarga Holm lebih kecil; paket ETHICS lama memakai randomisasi selisih rata-rata.
Jadi p-value dan banyaknya tanda signifikan lama–baru **tidak boleh dibandingkan seolah hanya data yang berubah**.
Ukuran mean_delta, median_delta, n_pairs, n_nonzero, dan rank-biserial tersedia untuk penelaahan.
Tidak dibuat klaim bahwa ambang k atau satu p-value menjamin kualitas pengukuran.

## Keterbatasan yang tetap berlaku
1. Validasi manusia independen kamus/parsing belum selesai; gambar bukan hasil final siap tesis.
2. I–E membandingkan paket persona dan konteks percakapan; implicit dapat memiliki warmup assistant panjang. Bukan efek identitas murni. Lihat conversation_context.csv.
3. Kesimpulan hanya untuk penjelasan tersedia; cakupan rendah bukan bukti model tidak memiliki suatu ciri bahasa.
4. ETHICS gabungan berbobot pasangan yang tersedia, bukan empat subset berbobot sama. Baca pair_support.csv dan pooled_subset_direction_check.csv; utamakan analisis subset.
5. Definisi tokenisasi/kamus antarbenchmark tidak sepenuhnya sama. Titik temu adalah 14 marker bahasa umum dan rancangan pelaporannya; marker spesifik tugas tidak dipaksakan identik.
6. Deontology pada eksperimen tidak memasukkan excuse; virtue menggunakan instruksi moral umum. Tidak ada perubahan scoring atau klaim akurasi gabungan.
7. Rujukan teori pada registry bukan bukti semua regex tervalidasi; kamus korpus/custom tetap kandidat.

## Tabel penting
- source_audit.csv: provenance tiap JSON, kanal, status pairing, dan konteks percakapan.
- pairing_audit.csv: perbandingan pairing nomor lama vs isi soal.
- response_records.csv.gz: seluruh pengukuran dan final text; tidak menyimpan reasoning tersembunyi.
- coverage.csv, pair_support.csv: denominator respons dan pasangan.
- outcome_registry.csv, paired_statistics.csv.gz: outcome, efek, dukungan, p mentah/Holm, metode, status CI.
- figure_index.csv: indeks semua PNG.
- manifest.json: hash sumber dan keterbatasan.
"""
(RUN/'README.md').write_text(notes,encoding='utf-8')
# Publish only after numerical, provenance and PNG checks pass. Preserve old assets.
archive=ACTIVE/'_arsip'/f'before_rebuild_{RUN_ID}'
archive.mkdir(parents=True,exist_ok=False)
active_figures=ACTIVE/'figures'
assert ACTIVE.resolve().is_relative_to((ROOT/'notebooks/religion/linguistic marker').resolve())
old_hashes={}
if active_figures.exists():
    old_hashes={str(p.relative_to(active_figures)):file_sha(p) for p in active_figures.rglob('*') if p.is_file()}
    shutil.move(str(active_figures),str(archive/'figures'))
    assert old_hashes=={str(p.relative_to(archive/'figures')):file_sha(p) for p in (archive/'figures').rglob('*') if p.is_file()}
shutil.copytree(FIGURES,active_figures)
for name in ['README.md','LATEST_RUN.json','figure_index.csv']:
    p=ACTIVE/name
    if p.exists():
        shutil.copy2(p,archive/name)
shutil.copy2(RUN/'README.md',ACTIVE/'README.md')
shutil.copy2(RUN/'figure_index.csv',ACTIVE/'figure_index.csv')
(ACTIVE/'LATEST_RUN.json').write_text(json.dumps({'run':str(RUN),'status':SCIENTIFIC_STATUS,'figures':len(figure_index),'archive':str(archive)},indent=2),encoding='utf-8')
(archive/'preserved_hashes.json').write_text(json.dumps(old_hashes,indent=2),encoding='utf-8')
shutil.copy2(ACTIVE/NOTEBOOK_NAME,RUN/'notebook_source.ipynb')
print('Pemeriksaan berhasil. PNG aktif:',active_figures)
print('Arsip lama:',archive)
print(json.dumps({k:summary[k] for k in ['rows','available_explanations','figures','original_id_question_mismatches_across_contrasts','status']},indent=2))
display(Image(filename=str(active_figures/f'00_cakupan_{SCOPES[0]}.png'),width=1200))
